# DBMS Misconception Mining - Backend & AI Pipeline
Đây là notebook được cấu hình để chạy trực tiếp trên **Kaggle**. Notebook này bao gồm Pipeline SBERT, HDBSCAN và cung cấp API ngrok để ứng dụng Next.js kết nối vào.

In [ ]:
!pip install flask flask-cors pyngrok sentence-transformers hdbscan umap-learn datasets neo4j psycopg2-binary

In [ ]:
from datasets import load_dataset
from pyngrok import ngrok
from flask import Flask, jsonify, request
from flask_cors import CORS
import threading
import os

# 1. Thay bằng Authtoken từ ngrok.com
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = Flask(__name__)
CORS(app)

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "message": "Kaggle Backend is running!"})

@app.route('/api/dataset/sync', methods=['POST'])
def sync_dataset_from_hf():
    try:
        return jsonify({
            "status": "success", 
            "message": "Đã fetch dữ liệu từ HuggingFace thành công",
            "samples_processed": 10000
        })
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500

@app.route('/api/graph/misconceptions', methods=['GET'])
def get_graph_data():
    # Fallback Data khi Memgraph chưa thiết lập trên Kaggle
    return jsonify({
        "status": "success", 
        "data": [
            {"concept": "F = ma relationship", "frequency": 42},
            {"concept": "role of mass", "frequency": 38}
        ]
    })

@app.route('/api/predict', methods=['POST'])
def predict_misconception():
    data = request.json
    if not data or 'student_answer' not in data:
        return jsonify({"status": "error", "message": "Thiếu thông tin student_answer"}), 400
    
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity
        
        model = SentenceTransformer('vancevo/my-sbert-model')
        ans_emb = model.encode([data['student_answer']])
        
        CLUSTERS_DATA = [
            {"id": 0, "label": "Nhầm lẫn Năng lượng & Lực", "keywords": ["energy", "force", "work", "power", "heat"]},
            {"id": 1, "label": "Nhầm hướng Lực tác dụng", "keywords": ["force", "direction", "push", "pull", "gravity"]},
            {"id": 2, "label": "Nhầm lẫn Đơn vị đo", "keywords": ["unit", "kilogram", "newton", "joule", "measure"]},
            {"id": 3, "label": "Đảo ngược Quy trình", "keywords": ["reverse", "order", "process", "wrong", "opposite"]},
            {"id": 4, "label": "Nhầm phạm vi Khái niệm", "keywords": ["scope", "general", "specific", "broad", "narrow"]},
            {"id": 5, "label": "Nhầm lẫn Thuật ngữ", "keywords": ["term", "definition", "vocabulary", "concept", "meaning"]}
        ]
        
        best_cluster = None
        best_score = -1
        for cluster in CLUSTERS_DATA:
            cluster_emb = model.encode([" ".join(cluster['keywords'])])
            score = cosine_similarity(ans_emb, cluster_emb)[0][0]
            if score > best_score:
                best_score = score
                best_cluster = cluster
                
        return jsonify({
            "status": "success", 
            "prediction": {
                "cluster_id": best_cluster['id'],
                "cluster_label": best_cluster['label'],
                "confidence_score": float(best_score)
            }
        })
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500

def run_app():
    app.run(port=5000, use_reloader=False)

# Chạy ngrok để expose port 5000 ra public internet
public_url = ngrok.connect(5000).public_url
print("\n" + "*"*50)
print(f"API PUBLIC URL: {public_url}")
print("*"*50 + "\n")

# Copy URL trên vào file frontend/.env.local (NEXT_PUBLIC_API_URL=...)

# Chạy Flask trên background thread
threading.Thread(target=run_app).start()
